# Random Forest — multibranch-style sequence classifier

This notebook uses the cleaned `base_utils_qwen.py` and trains a sequence-level Random Forest.

Local quick runs use `data/sample.csv` (37 sequences). Set `use_sample_data = False` for full `train.csv`.

Switch `search_mode` between `'grid'` and `'bayesian'` in the config cell.

Style:
- configuration
- data loading
- split
- estimator
- parameter search (grid or Bayesian)
- holdout evaluation
- save results


In [ ]:
import os
import sys
import warnings
import importlib
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Local path routing (works from notebooks/ or project root)
current_dir = os.getcwd()
workspace_root = current_dir
if os.path.basename(current_dir) == 'notebooks':
    workspace_root = os.path.dirname(current_dir)

src_path = os.path.join(workspace_root, 'src')
sys.path.insert(0, workspace_root)
sys.path.insert(0, src_path)

# Kaggle path routing
try:
    dataset_name = os.listdir('/kaggle/input/datasets/keithmarange')[0]
    sys.path.append(f'/kaggle/input/datasets/keithmarange/{dataset_name}/')
    sys.path.append('/kaggle/input/cmi-competition-code')
except Exception:
    pass

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, GroupKFold, GroupShuffleSplit
from sklearn.metrics import f1_score, make_scorer
from sklearn.pipeline import Pipeline

try:
    import skopt
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
    SKOPT_AVAILABLE = True
except ImportError:
    BayesSearchCV = None
    Categorical = Integer = Real = None
    SKOPT_AVAILABLE = False

try:
    import src.base_utils_qwen as base_utils_qwen
    importlib.reload(base_utils_qwen)
    from src import data_utils
    from src.base_utils_qwen import (
        SequenceExtractor,
        RandomForestSequenceClassifier,
        competition_scorer,
        evaluate_holdout,
        make_competition_scorer,
        prepare_bayesian_space,
        SensorAugmentor
    )
    print('Imports loaded from src/ (reloaded)')
except ImportError:
    import data_utils
    import importlib
    importlib.reload(sys.modules.get('base_utils_qwen', __import__('base_utils_qwen')))
    from base_utils_qwen import (
        SequenceExtractor,
        RandomForestSequenceClassifier,
        competition_scorer,
        evaluate_holdout,
        make_competition_scorer,
        prepare_bayesian_space,
        SensorAugmentor
    )
    print('Imports loaded from flat src path (reloaded)')


In [ ]:
# Install optional search / feature dependencies if missing (safe to re-run)
for package_name, import_name in [
    ('scikit-optimize', 'skopt'),
    ('PyWavelets', 'pywt'),
]:
    try:
        __import__(import_name)
    except ImportError:
        import subprocess
        import sys
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package_name])
        print(f'Installed {package_name}')

try:
    import skopt
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
    SKOPT_AVAILABLE = True
    print(f'scikit-optimize {skopt.__version__} ready')
except ImportError:
    BayesSearchCV = None
    Categorical = Integer = Real = None
    SKOPT_AVAILABLE = False
    print('scikit-optimize unavailable')

try:
    import pywt
    print(f'PyWavelets {pywt.__version__} ready')
except ImportError:
    print('PyWavelets unavailable')

In [ ]:
experiment_name = 'random_forest'

TARGET_COL = 'bfrb'

# Use sample.csv for a quick smoke test; set False for full train.csv
use_sample_data = False
sample_file = 'sample.csv'  # also available: eg.csv

search_mode = 'grid'  # 'grid' or 'bayesian'; Bayesian smoke test below uses sample data
random_state = 42
n_splits = 1          # 1 -> GroupShuffleSplit; >=2 -> GroupKFold
cv_test_size = 0.4
train_size = 0.6
n_iter = 13            # Bayesian iterations for a quick smoke test
verbose = 3
error_score = 'raise'  # 'raise' or 'warn'

orientation_filter_list = [] # ['Seated Lean Non Dom - FACE DOWN', 'Lie on Side - Non Dominant', 'Seated Straight', 'Lie on Back' ]

problematic_sequence_bool = True
target_only_bool = False

results_dir = Path(f'results_{experiment_name}')
results_dir.mkdir(exist_ok=True)
timestamp = datetime.now().strftime('%Y%m%d_%H%M')

if TARGET_COL == 'bfrb':
    scorer = competition_scorer
else:
    scorer = make_scorer(f1_score, average='macro', zero_division=0)

search_mode = str(search_mode).lower()
if search_mode in ('bayes', 'bayesian'):
    search_mode = 'bayesian'
elif search_mode != 'grid':
    raise ValueError("search_mode must be 'grid' or 'bayesian'")

if n_splits <= 1:
    cv_object = GroupShuffleSplit(
        n_splits=1,
        test_size=cv_test_size,
        random_state=random_state,
    )
else:
    cv_object = GroupKFold(n_splits=n_splits)


In [ ]:
data_root = data_utils.find_data_root()
sample_path = data_root / sample_file

if use_sample_data and sample_path.exists():
    raw_train_df = pd.read_csv(sample_path)
    print(f'Using {sample_file}: {raw_train_df["sequence_id"].nunique()} sequences')
else:
    raw_train_df = pd.read_csv(data_root / 'train.csv')
    print(f'Using train.csv: {raw_train_df["sequence_id"].nunique()} sequences')

train_demo_df = pd.read_csv(data_root / 'train_demographics.csv')

train_df = raw_train_df.set_index('row_id').copy(deep=True)

train_df['gesture'] = train_df['gesture'].fillna('non_bfrb').astype(str)
train_df['orientation'] = train_df['orientation'].fillna('Unknown').astype(str)
train_df['is_target'] = train_df['sequence_type'].eq('Target').astype(int)
train_df['bfrb'] = train_df['gesture'].where(train_df['is_target'].astype(bool), 'non_bfrb')

train_df['gesture_position'] = train_df['gesture'].str.split(' - ').str[0]
train_df['gesture_action'] = train_df['gesture'].str.split(' - ').str[-1]

problematic_sequence_df = train_df.groupby('sequence_id')[['acc_x', 'acc_y', 'acc_z', 'rot_x', 'rot_y', 'rot_w', 'rot_z']].skew().abs()
ideal_skew_threshold = 1.35
problematic_features_threshold = 6
result_series = ((problematic_sequence_df > ideal_skew_threshold).sum(axis=1) >= problematic_features_threshold)
problematic_sequences_list = result_series.loc[result_series].index
train_df['problematic_sequence'] = train_df['sequence_id'].isin(problematic_sequences_list).astype(bool)

if orientation_filter_list:
    train_df = train_df[train_df['orientation'].isin(orientation_filter_list)].copy()

if target_only_bool:
    train_df = train_df[train_df['is_target']].copy()

train_df[TARGET_COL] = train_df[TARGET_COL].fillna('non_bfrb').astype(str)


In [ ]:
try:
    train_sample_df, hold_out_df = data_utils.sample_balanced_split(
        train_df,
        train_pct=train_size,
        test_pct=min(0.2, 1 - train_size),
        random_state=random_state,
    )
except Exception:
    seq_df = train_df[['sequence_id', 'is_target', TARGET_COL]].drop_duplicates('sequence_id').sort_values('sequence_id')
    gss = GroupShuffleSplit(n_splits=1, train_size=train_size, random_state=random_state)
    train_idx, test_idx = next(gss.split(seq_df, groups=seq_df['sequence_id']))

    train_seqs = seq_df.iloc[train_idx]['sequence_id']
    test_seqs = seq_df.iloc[test_idx]['sequence_id']

    train_sample_df = train_df[train_df['sequence_id'].isin(train_seqs)].copy()
    hold_out_df = train_df[train_df['sequence_id'].isin(test_seqs)].copy()

X_train = train_sample_df.copy()

if problematic_sequence_bool:
    X_train = X_train[~X_train['problematic_sequence']].copy()
    
X_test = hold_out_df.copy()

y_train = X_train[['sequence_id', 'is_target', TARGET_COL]].copy()
y_test = X_test[['sequence_id', 'is_target', TARGET_COL]].copy()

groups = X_train['sequence_id'].astype(str)

print('Train sequences:', X_train['sequence_id'].nunique())
print('Test sequences:', X_test['sequence_id'].nunique())


In [ ]:
rf_pipeline = Pipeline([
    ('augmentor', SensorAugmentor(
        sequence_col='sequence_id',
        counter_col='sequence_counter',
        prob=0.0,
        per_aug_prob=0.0,
    )),
    ('estimator', RandomForestSequenceClassifier(
        primary_target=TARGET_COL,
        extractor=SequenceExtractor(
            output_format='frame',
            acc_modes='raw|velocity|jerk',
            rotation_modes='quaternion|angular_velocity',
        ),
        estimator=RandomForestClassifier(
            n_estimators=300,
            random_state=random_state,
        ),
        random_state=random_state,
    )),
])

In [ ]:
# ============================================================
# GRID SEARCH SPACE
# Use the nested parameter names expected by the pipeline:
#   - augmentor__...
#   - estimator__extractor__...
#   - estimator__estimator__...
# ============================================================

GRID_PARAM_SPACE = {
    # ------------------------------------------------------------
    # AUGMENTOR
    # ------------------------------------------------------------
    'augmentor__prob': [0.0],
    'augmentor__sensor_drop_prob': [0.0],
    'augmentor__noise_std': [0.0],
    'augmentor__jitter_sigma': [0.0],
    'augmentor__scaling_sigma': [0.0],
    'augmentor__channel_drop_prob': [0.0],
    'augmentor__time_shift_frac': [0.0],
    'augmentor__temporal_num_masks': [0],
    'augmentor__temporal_mask_frac': [0.0],

    # ------------------------------------------------------------
    # INNER SequenceExtractor parameters
    # ------------------------------------------------------------
    'estimator__extractor__acc_modes': ['smoothed|velocity|displacement|jerk'],
    'estimator__extractor__rotation_modes': ['quaternion|euler|angular_velocity'],
    'estimator__extractor__tof_modes': ['pooled_stats|sensor_stats'],
    'estimator__extractor__thm_modes': ['centered_diff'],
    'estimator__extractor__frame_stats': ['mean,std,min,max,last,first,rms'],
    'estimator__extractor__motion_filter_mode': [None],
    'estimator__extractor__use_dead_reckoning': [False],
    'estimator__extractor__dead_reckoning_detrend': [False],
    'estimator__extractor__kalman_process_noise': [1e-3],
    'estimator__extractor__kalman_measurement_noise': [1e-1],
    'estimator__extractor__window_size': [5],
    'estimator__extractor__smooth_alpha': [None],
    'estimator__extractor__clip_value': [None],
    'estimator__extractor__interp_mode': ['linear'],
    'estimator__extractor__output_format': ['frame'],
    'estimator__extractor__padding_value': [0.0],
    'estimator__extractor__maxlen': [160],
    'estimator__extractor__chunk_window_size': [100],
    'estimator__extractor__chunk_stride': [50],
    'estimator__extractor__add_global_context': [False, True],
    'estimator__extractor__resample_modalities': [False, True],
    'estimator__extractor__compute_dt': [True],
    'estimator__extractor__imu_native_sampling_rate': [100],
    'estimator__extractor__rot_native_sampling_rate': [100],
    'estimator__extractor__tof_native_sampling_rate': [20],
    'estimator__extractor__thm_native_sampling_rate': [20],
    'estimator__extractor__imu_target_sampling_rate': [100],
    'estimator__extractor__rot_target_sampling_rate': [100],
    'estimator__extractor__tof_target_sampling_rate': [20],
    'estimator__extractor__thm_target_sampling_rate': [20],
    'estimator__extractor__stft_nperseg': [None, 100],
    'estimator__extractor__stft_noverlap': [50],
    'estimator__extractor__stft_window_type': ['hann'],
    'estimator__extractor__stft_use_log_scale': [True],
    'estimator__extractor__cwt_wavelet': ['morl'],
    'estimator__extractor__cwt_max_scale': [32],
    'estimator__extractor__cwt_n_scales': [32],
    'estimator__extractor__cwt_use_log_scale': [False],

    # ------------------------------------------------------------
    # INNER RandomForestClassifier parameters
    # ------------------------------------------------------------
    'estimator__estimator__n_estimators': [37, 200],
    'estimator__estimator__criterion': ['gini'],
    'estimator__estimator__max_depth': [40],
    'estimator__estimator__min_samples_split': [20],
    'estimator__estimator__min_samples_leaf': [20],
    'estimator__estimator__max_features': ['sqrt'],
    'estimator__estimator__bootstrap': [True],
    'estimator__estimator__class_weight': ['balanced'],
    'estimator__estimator__max_leaf_nodes': [None],
    'estimator__estimator__ccp_alpha': [0.0, 10.0],
    'estimator__estimator__max_samples': [None],
}


# ============================================================
# BAYESIAN SEARCH SPACE
# Full exploration space using the same nesting convention.
# ============================================================

if SKOPT_AVAILABLE:
    try:
        BAYESIAN_PARAM_SPACE = {
            # AUGMENTOR
            'augmentor__prob': Categorical([0.0]),
            'augmentor__sensor_drop_prob': Real(0.0, 0.5),
            'augmentor__noise_std': Categorical([0.0]),
            'augmentor__jitter_sigma': Categorical([0.0]),
            'augmentor__scaling_sigma': Categorical([0.0]),
            'augmentor__channel_drop_prob': Categorical([0.0]),
            'augmentor__time_shift_frac': Categorical([0.0]),
            'augmentor__temporal_num_masks': Categorical([0]),
            'augmentor__temporal_mask_frac': Categorical([0.0]),

            # SequenceExtractor inside RandomForestSequenceClassifier
            'estimator__extractor__acc_modes': Categorical([
                'raw',
                'raw|velocity',
                'smoothed|velocity|displacement|jerk',
            ]),
            'estimator__extractor__rotation_modes': Categorical([
                'quaternion|euler|angular_velocity',
                'quaternion|angular_velocity|delta_euler|rot6d',
            ]),
            'estimator__extractor__tof_modes': Categorical([
                'pooled_stats|sensor_stats',
            ]),
            'estimator__extractor__thm_modes': Categorical([
                'centered_diff',
            ]),
            'estimator__extractor__frame_stats': Categorical([
                'mean,std,min,max,last',
                'mean,std,min,max,last,first,rms,abs_mean',
            ]),
            'estimator__extractor__motion_filter_mode': Categorical([
                'extended_kalman',
            ]),
            'estimator__extractor__use_dead_reckoning': Categorical([True]),
            'estimator__extractor__dead_reckoning_detrend': Categorical([True]),
            'estimator__extractor__kalman_process_noise': Real(1e-5, 1e-1, prior='log-uniform'),
            'estimator__extractor__kalman_measurement_noise': Real(1e-3, 1e1, prior='log-uniform'),
            'estimator__extractor__window_size': Integer(5, 50),
            'estimator__extractor__smooth_alpha': Categorical([
                None,
                0.05,
                0.10,
                0.20,
                0.90,
            ]),
            'estimator__extractor__clip_value': Categorical([150.0]),
            'estimator__extractor__interp_mode': Categorical(['linear']),
            'estimator__extractor__stft_nperseg': Categorical([32, 64, 128]),
            'estimator__extractor__stft_noverlap': Categorical([None, 8, 16, 32]),
            'estimator__extractor__stft_window_type': Categorical(['hann', 'hamming', 'blackman']),
            'estimator__extractor__stft_use_log_scale': Categorical([True, False]),
            'estimator__extractor__cwt_wavelet': Categorical(['morl', 'mexh', 'gaus1', 'gaus2']),
            'estimator__extractor__cwt_max_scale': Categorical([64, 128, 256]),
            'estimator__extractor__cwt_n_scales': Categorical([16, 32, 64]),
            'estimator__extractor__cwt_use_log_scale': Categorical([True, False]),
            'estimator__extractor__output_format': Categorical(['frame']),
            'estimator__extractor__padding_value': Categorical([0.0]),
            'estimator__extractor__maxlen': Categorical([150]),
            'estimator__extractor__chunk_window_size': Categorical([None, 100]),
            'estimator__extractor__chunk_stride': Categorical([None, 50]),
            'estimator__extractor__add_global_context': Categorical([True]),
            'estimator__extractor__compute_dt': Categorical([True]),
            'estimator__extractor__imu_native_sampling_rate': Categorical([100]),
            'estimator__extractor__rot_native_sampling_rate': Categorical([100]),
            'estimator__extractor__tof_native_sampling_rate': Categorical([20]),
            'estimator__extractor__thm_native_sampling_rate': Categorical([20]),
            'estimator__extractor__imu_target_sampling_rate': Categorical([100]),
            'estimator__extractor__rot_target_sampling_rate': Categorical([100]),
            'estimator__extractor__tof_target_sampling_rate': Categorical([20]),
            'estimator__extractor__thm_target_sampling_rate': Categorical([20]),
            'estimator__extractor__resample_modalities': Categorical([True]),

            # RandomForestClassifier inside RandomForestSequenceClassifier
            'estimator__estimator__n_estimators': Integer(5, 300),
            'estimator__estimator__criterion': Categorical(['gini', 'entropy']),
            'estimator__estimator__max_depth': Categorical([None, 10, 30, 50, 100]),
            'estimator__estimator__min_samples_split': Integer(2, 20),
            'estimator__estimator__min_samples_leaf': Integer(1, 8),
            'estimator__estimator__max_features': Categorical([0.3, 0.5, 0.7]),
            'estimator__estimator__bootstrap': Categorical([True]),
            'estimator__estimator__class_weight': Categorical(['balanced']),
            'estimator__estimator__min_impurity_decrease': Real(0.0, 0.005),
            'estimator__estimator__max_leaf_nodes': Categorical([None]),
            'estimator__estimator__ccp_alpha': Categorical([0.0]),
            'estimator__estimator__max_samples': Categorical([None]),
        }
    except Exception:
        BAYESIAN_PARAM_SPACE = GRID_PARAM_SPACE
else:
    BAYESIAN_PARAM_SPACE = GRID_PARAM_SPACE

param_space = BAYESIAN_PARAM_SPACE if search_mode == 'bayesian' else GRID_PARAM_SPACE


In [ ]:
if search_mode == 'bayesian':
    if not SKOPT_AVAILABLE:
        raise ImportError(
            "Bayesian search requires scikit-optimize. "
            "Install with: pip install scikit-optimize"
        )

    search = BayesSearchCV(
        estimator=rf_pipeline,
        search_spaces=param_space,
        n_iter=n_iter,
        scoring=scorer,
        cv=cv_object,
        n_jobs=1,
        random_state=random_state,
        verbose=verbose,
        return_train_score=True,
        error_score=error_score,
    )
else:
    search = GridSearchCV(
        estimator=rf_pipeline,
        param_grid=param_space,
        scoring=scorer,
        cv=cv_object,
        n_jobs=1,
        verbose=verbose,
        return_train_score=True,
        error_score=error_score,
    )

search.fit(X_train, y_train, groups=groups)

print('Best CV score:', search.best_score_)
print('Best params:', search.best_params_)


In [ ]:
best_model = search.best_estimator_

y_pred = best_model.predict(X_test)

eval_results = evaluate_holdout(
    y_test,
    y_pred,
    target_col=TARGET_COL,
    verbose=True,
)

print('Holdout competition score:', eval_results['competition_score'])

cv_df = pd.DataFrame(search.cv_results_)
cv_df.to_csv(results_dir / f'rf_multibranch_style_cv_{timestamp}.csv', index=False)

eval_results['results_df'].to_csv(
    results_dir / f'rf_multibranch_style_holdout_{timestamp}.csv',
    index=False,
)

pd.DataFrame(
    [
        {
            'best_score': search.best_score_,
            'best_params': str(search.best_params_),
            'holdout_score': eval_results['competition_score'],
        }
    ]
).to_csv(
    results_dir / f'rf_multibranch_style_best_{timestamp}.csv',
    index=False,
)


In [ ]:
best_estimator = best_model.named_steps['estimator']
importances = pd.Series(
    best_estimator.estimator_.feature_importances_,
    index=best_estimator.extractor_.frame_feature_names_,
).sort_values(ascending=False)

print(importances.head(50))